In [6]:
# ============================
# 📗 TF–IDF from PDF by PARAGRAPH
# ============================

# ---- 1. Install dependencies ----
!pip install PyPDF2 scikit-learn pandas openpyxl --quiet

# ---- 2. Upload PDF ----
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# ---- 3. Extract text paragraph-wise ----
from PyPDF2 import PdfReader
import re

def extract_paragraphs(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    # Split into paragraphs (blank lines or double newlines)
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if len(p.strip()) > 30]
    return paragraphs

paragraphs = extract_paragraphs(pdf_path)
print(f"✅ Extracted {len(paragraphs)} paragraphs from PDF.")

# ---- 4. Compute TF, IDF, TF-IDF ----
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

# Preprocessing: remove digits
def custom_tokenizer(text):
    text = re.sub(r'\d+', '', text)           # remove numbers
    tokens = re.findall(r'\b[a-zA-Z]{3,}\b', text.lower())  # words >=3 letters
    return tokens

vectorizer = TfidfVectorizer(tokenizer=custom_tokenizer, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(paragraphs)
feature_names = vectorizer.get_feature_names_out()

# Get TF, IDF, TF-IDF
idf_values = vectorizer.idf_
tf_values = np.asarray(tfidf_matrix.mean(axis=0)).ravel() / idf_values  # approximate mean TF
tfidf_values = tfidf_matrix.mean(axis=0).ravel().tolist()[0] if hasattr(tfidf_matrix, 'mean') else tf_values * idf_values

# Compute TF-IDF average per term
tfidf_mean = np.asarray(tfidf_matrix.mean(axis=0)).ravel()

# Combine all into DataFrame
df = pd.DataFrame({
    'Keyword': feature_names,
    'TF': np.round(tf_values, 6),
    'IDF': np.round(idf_values, 6),
    'TF-IDF': np.round(tfidf_mean, 6)
})

# Sort by TF-IDF descending and select top 100
df_top100 = df.sort_values(by='TF-IDF', ascending=False).head(100).reset_index(drop=True)

# ---- 5. Export to Excel ----
excel_name = pdf_path.replace('.pdf', '_tfidf_top100.xlsx')
df_top100.to_excel(excel_name, index=False)
files.download(excel_name)

print(f"✅ Top 100 keywords exported to '{excel_name}'")


Saving 1-99 Bottles of OOP- A Practical Guide to Object-Oriented -- Sandi Metz, Katrina Owen, TJ Stankus -- ( WeLib.org ).pdf to 1-99 Bottles of OOP- A Practical Guide to Object-Oriented -- Sandi Metz, Katrina Owen, TJ Stankus -- ( WeLib.org ) (1).pdf
✅ Extracted 14 paragraphs from PDF.


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Top 100 keywords exported to '1-99 Bottles of OOP- A Practical Guide to Object-Oriented -- Sandi Metz, Katrina Owen, TJ Stankus -- ( WeLib.org ) (1)_tfidf_top100.xlsx'
